# Deepfake hackathon: T4 timing + reproducibility run

Settings (right panel): **Accelerator = GPU T4 x2**, **Internet = On** (needed only for the install cell),
input = your private dataset made from `kaggle_kit.zip` (see `scripts/kaggle/README.md`).
Run all cells top to bottom, then download `/kaggle/working/t4_results.zip` (Output tab, or the link printed by the last cell).

In [ ]:
!nvidia-smi
import glob, os
hits = glob.glob('/kaggle/input/**/kit_manifest.json', recursive=True)
assert hits, 'kit not found: attach the dataset (Add Input) and check it finished unpacking'
KIT = os.path.dirname(hits[0])
print('KIT =', KIT)
print(open(f'{KIT}/kit_manifest.json').read()[:600])

## Python 3.10 venv + pinned inference requirements
`uv` downloads a standalone CPython 3.10 (the brief's version); the venv then gets exactly `requirements-infer.txt`
(torch 2.11.0+cu128 etc.) via pip, as the organisers would install it. Takes ~2-4 min (torch wheel ~1 GB).

In [ ]:
!pip install -q uv
!uv venv --python 3.10 --seed /kaggle/working/venv
!/kaggle/working/venv/bin/pip install -q -r {KIT}/requirements-infer.txt
!/kaggle/working/venv/bin/python -c "import sys, torch, timm; print(sys.version); print(torch.__version__, torch.version.cuda, torch.cuda.get_arch_list(), torch.cuda.get_device_name(0), torch.cuda.get_device_capability(0)); print('timm', timm.__version__)"

## Timed run (x2), validator, determinism, diff vs local CSV
Wall time = fresh `python inference_script.py ...` process, including imports and model load. Run 1 = cold cache.
The script exits non-zero (and prints `hard failures`) if anything must be fixed before submitting.

In [ ]:
!/kaggle/working/venv/bin/python {KIT}/t4_run.py --kit_dir /kaggle/input --out /kaggle/working/t4_out

In [ ]:
print(open('/kaggle/working/t4_out/t4_report.md').read())
!cd /kaggle/working && zip -qr t4_results.zip t4_out && ls -la t4_results.zip
from IPython.display import FileLink
FileLink('t4_results.zip')